### Data Loading and exploration

Each support ticket is treated as a semantic unit.

The embedding text is composed of the ticket title and description,
while the remaining fields are preserved as metadata for ChromaDB.

In [1]:
import pandas as pd

In [2]:
DATA_PATH = "../data/it_support_tickets.csv"

tickets_df = pd.read_csv(DATA_PATH)

tickets_df.head()

,ticket_id,title,description,category,priority,status
0,1,Error 403 al ejecutar GitHub Action,El workflow de GitHub devuelve un error 403 cu...,GitHub,high,open
1,2,Token sin permisos suficientes,La aplicación puede consultar GitHub pero reci...,GitHub,high,open
2,3,Error de autenticación al iniciar sesión,El usuario introduce credenciales válidas pero...,authentication,high,open
3,4,Sesión expirada inesperadamente,La sesión del usuario termina antes del tiempo...,authentication,medium,open
4,5,Contraseña olvidada,Un usuario no recuerda su contraseña y necesit...,authentication,low,closed


In [3]:
tickets_df.shape

(35, 6)

In [4]:
tickets_df.columns

Index(['ticket_id', 'title', 'description', 'category', 'priority', 'status'], dtype='str')

In [5]:
tickets_df["category"].value_counts()

category
authentication    5
frontend          5
deployment        5
database          4
backend           3
GitHub            2
performance       2
integration       2
CI/CD             1
testing           1
quality           1
observability     1
cloud             1
security          1
dependencies      1
Name: count, dtype: int64

In [6]:
tickets_df[["title", "description"]].head(10)

,title,description
0,Error 403 al ejecutar GitHub Action,El workflow de GitHub devuelve un error 403 cu...
1,Token sin permisos suficientes,La aplicación puede consultar GitHub pero reci...
2,Error de autenticación al iniciar sesión,El usuario introduce credenciales válidas pero...
3,Sesión expirada inesperadamente,La sesión del usuario termina antes del tiempo...
4,Contraseña olvidada,Un usuario no recuerda su contraseña y necesit...
5,Error 500 en el dashboard,El dashboard muestra un error interno del serv...
6,API devuelve datos incorrectos,Un endpoint de la aplicación responde correcta...
7,Endpoint responde demasiado lento,Una consulta a la API tarda varios segundos en...
8,Aplicación pierde conexión con PostgreSQL,El backend pierde periódicamente la conexión c...
9,Consulta PostgreSQL muy lenta,Una operación que consulta muchos registros ta...


### Chunking strategy

Each support ticket is treated as a semantic unit.

The embedding text is composed of the ticket title and description,
while the remaining fields are preserved as metadata for ChromaDB.

In [7]:
def create_chunk(row):
    return (
        f"Title: {row['title']}\n"
        f"Description: {row['description']}"
    )

tickets_df["chunk"] = tickets_df.apply(create_chunk, axis=1)

In [8]:
print(tickets_df.loc[0, "chunk"])

Title: Error 403 al ejecutar GitHub Action
Description: El workflow de GitHub devuelve un error 403 cuando un usuario intenta ejecutarlo desde la aplicación.


### Embedding step

In [9]:
import ollama

#### Sample of embedding first chunk

In [10]:
response = ollama.embed(
    model="nomic-embed-text",
    input=tickets_df.loc[0, "chunk"]
)

In [11]:
type(response)

ollama._types.EmbedResponse

In [12]:
embedding = response["embeddings"][0]

len(embedding)

768

In [15]:
embedding[:10]

[0.062743194,
 0.05472684,
 -0.1429663,
 -0.044084225,
 0.054838907,
 -0.014185818,
 -0.007555809,
 -0.022642922,
 -0.030183528,
 -0.015195298]

#### Embedding all data

In [17]:
embeddings = []

for chunk in tickets_df["chunk"]:
    response = ollama.embeddings(
        model="nomic-embed-text",
        prompt=chunk
    )

    embeddings.append(response["embedding"])

tickets_df["embedding"] = embeddings

In [18]:
len(tickets_df["embedding"])

35

In [19]:
set(len(embedding) for embedding in tickets_df["embedding"])

{768}

### chromaDB load data

In [20]:
import chromadb

In [21]:
client = chromadb.PersistentClient(
    path="../chroma_db"
)

In [22]:
collection = client.get_or_create_collection(
    name="it_support_tickets"
)

In [23]:
ids = tickets_df["ticket_id"].astype(str).tolist()

documents = tickets_df["chunk"].tolist()

embeddings = tickets_df["embedding"].tolist()

In [24]:
metadatas = tickets_df[
    ["category", "priority", "status"]
].to_dict(orient="records")

In [25]:
print(ids[0])
print(documents[0])
print(embeddings[0][:5])
print(metadatas[0])

1
Title: Error 403 al ejecutar GitHub Action
Description: El workflow de GitHub devuelve un error 403 cuando un usuario intenta ejecutarlo desde la aplicación.
[1.3267537355422974, 1.157241702079773, -3.0231339931488037, -0.9321953058242798, 1.159611463546753]
{'category': 'GitHub', 'priority': 'high', 'status': 'open'}


In [29]:
if collection.count() == 0:
    collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas)

In [30]:
collection.count()

35

### chromaDB sample query

In [41]:
query_text = "No puedo ejecutar un workflow de GitHub porque no tengo permisos suficientes."

#### embedding query

In [42]:
query_response = ollama.embeddings(
    model="nomic-embed-text",
    prompt=query_text
)

query_embedding = query_response["embedding"]

In [43]:
len(query_embedding)

768

#### search query in collection

In [44]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

In [45]:
results.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])

In [46]:
results["ids"]

[['1', '17', '2', '12', '23']]

In [47]:
results["distances"]

[[211.09523010253906,
  221.7834014892578,
  228.24319458007812,
  264.3489990234375,
  283.137939453125]]

In [48]:
results["metadatas"]

[[{'priority': 'high', 'status': 'open', 'category': 'GitHub'},
  {'category': 'CI/CD', 'status': 'open', 'priority': 'medium'},
  {'priority': 'high', 'status': 'open', 'category': 'GitHub'},
  {'priority': 'medium', 'status': 'open', 'category': 'frontend'},
  {'status': 'open', 'category': 'authentication', 'priority': 'high'}]]

In [49]:
for i in range(5):
    print(f"Result {i + 1}")
    print("ID:", results["ids"][0][i])
    print("Distance:", results["distances"][0][i])
    print("Metadata:", results["metadatas"][0][i])
    print("Document:", results["documents"][0][i])
    print("-" * 80)

Result 1
ID: 1
Distance: 211.09523010253906
Metadata: {'priority': 'high', 'status': 'open', 'category': 'GitHub'}
Document: Title: Error 403 al ejecutar GitHub Action
Description: El workflow de GitHub devuelve un error 403 cuando un usuario intenta ejecutarlo desde la aplicación.
--------------------------------------------------------------------------------
Result 2
ID: 17
Distance: 221.7834014892578
Metadata: {'category': 'CI/CD', 'status': 'open', 'priority': 'medium'}
Document: Title: GitHub Actions falla en CI
Description: El pipeline de integración continua falla durante la ejecución de las pruebas automatizadas.
--------------------------------------------------------------------------------
Result 3
ID: 2
Distance: 228.24319458007812
Metadata: {'priority': 'high', 'status': 'open', 'category': 'GitHub'}
Document: Title: Token sin permisos suficientes
Description: La aplicación puede consultar GitHub pero recibe un error de autorización al ejecutar determinadas acciones.
----

### cosine similarity

In [52]:
from sklearn.metrics.pairwise import cosine_similarity

In [60]:
document_high_similarity_id_sample = 1

In [61]:
ticket_1_embedding = tickets_df.loc[
    tickets_df["ticket_id"] == document_high_similarity_id_sample,
    "embedding"
].iloc[0]

In [62]:
similarity = cosine_similarity(
    [query_embedding],
    [ticket_1_embedding]
)

similarity

array([[0.7475537]])

**En qué se relaciona 0.7475 (cosine similarity) con 211 (chromaDB distance) ?**

In [72]:
collection.configuration["hnsw"]

{'space': 'l2',
 'ef_construction': 100,
 'ef_search': 100,
 'max_neighbors': 16,
 'resize_factor': 1.2,
 'sync_threshold': 1000}

_ChromaDB utiliza por defecto L2 (distancia euclídea al cuadrado) en lugar de similitud por cosenos._

### Test ChromaDB collection with Cosine similarity metric configuration

In [70]:
cosine_collection = client.get_or_create_collection(
    name="it_support_tickets_cosine",
    configuration={
        "hnsw": {
            "space": "cosine"
        }
    }
)

In [71]:
cosine_collection.configuration

{'hnsw': {'space': 'cosine',
  'ef_construction': 100,
  'ef_search': 100,
  'max_neighbors': 16,
  'resize_factor': 1.2,
  'sync_threshold': 1000},
 'spann': None,
 'embedding_function': <chromadb.api.types.DefaultEmbeddingFunction at 0x7763e7097fe0>}

In [73]:
cosine_collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas
)

In [74]:
cosine_collection.count()

35

In [75]:
results_cosine = cosine_collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

In [76]:
for i in range(5):
    print(f"Result {i + 1}")
    print("ID:", results_cosine["ids"][0][i])
    print("Distance:", results_cosine["distances"][0][i])
    print("Metadata:", results_cosine["metadatas"][0][i])
    print("Document:", results_cosine["documents"][0][i])
    print("-" * 80)

Result 1
ID: 1
Distance: 0.2524462938308716
Metadata: {'category': 'GitHub', 'priority': 'high', 'status': 'open'}
Document: Title: Error 403 al ejecutar GitHub Action
Description: El workflow de GitHub devuelve un error 403 cuando un usuario intenta ejecutarlo desde la aplicación.
--------------------------------------------------------------------------------
Result 2
ID: 17
Distance: 0.26875829696655273
Metadata: {'status': 'open', 'priority': 'medium', 'category': 'CI/CD'}
Document: Title: GitHub Actions falla en CI
Description: El pipeline de integración continua falla durante la ejecución de las pruebas automatizadas.
--------------------------------------------------------------------------------
Result 3
ID: 2
Distance: 0.28006821870803833
Metadata: {'priority': 'high', 'status': 'open', 'category': 'GitHub'}
Document: Title: Token sin permisos suficientes
Description: La aplicación puede consultar GitHub pero recibe un error de autorización al ejecutar determinadas acciones.
-

___
**NOTA:** Estas distancias se interpretan como:
cosine distance = 1 - cosine similarity